# Análise Exploratória dos Dados de Ações

Este notebook realiza a análise exploratória dos dados de ações brasileiras e americanas coletados via `yfinance` e persistidos em tabelas Delta.

## Tabelas Disponíveis (Camada Gold)
- **`main.default.gold_resumo_ticker`** — Resumo agregado por ticker (preço médio, retorno total, volatilidade, ranking)
- **`main.default.gold_resumo_mensal`** — Resumo agregado mensal (fechamento, retorno mensal, volatilidade por mês)

## Tickers Analisados
- **Brasileiras (B3):** PETR4.SA, VALE3.SA, ITUB4.SA, BBAS3.SA, WEGE3.SA, B3SA3.SA  
- **Americanas:** AAPL, MSFT, NVDA, GOOGL, AMZN, META

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de visualização
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11

# Carregar dados da camada Gold (tabelas agregadas)
df_gold_ticker = spark.table("main.default.gold_resumo_ticker").toPandas()
df_gold_mensal = spark.table("main.default.gold_resumo_mensal").toPandas()

print(f"Resumo por ticker: {len(df_gold_ticker)} tickers")
print(f"Resumo mensal: {len(df_gold_mensal):,} registros")
print(f"Colunas Gold Ticker: {list(df_gold_ticker.columns)}")
print(f"Colunas Gold Mensal: {list(df_gold_mensal.columns)}")
df_gold_ticker.head(12)

In [0]:
print("=== Esquema Gold Resumo Ticker ===")
print(df_gold_ticker.dtypes)
print()
print("=== Esquema Gold Resumo Mensal ===")
print(df_gold_mensal.dtypes)
print()
print("=== Período de Dados (Gold Mensal) ===")
ano_mes_min = df_gold_mensal.loc[df_gold_mensal[["ano", "mes"]].sum(axis=1).idxmin()]
ano_mes_max = df_gold_mensal.loc[df_gold_mensal[["ano", "mes"]].sum(axis=1).idxmax()]
print(f"Data mínima: {int(ano_mes_min['ano'])}-{int(ano_mes_min['mes']):02d}")
print(f"Data máxima: {int(ano_mes_max['ano'])}-{int(ano_mes_max['mes']):02d}")
print(f"Tickers únicos: {df_gold_mensal['ticker'].nunique()}")
print(f"Tickers: {sorted(df_gold_mensal['ticker'].unique())}")

In [0]:
print("=== Estatísticas Descritivas — Gold Resumo Ticker ===")
display(df_gold_ticker.describe().round(2))

print("\n=== Resumo por Ticker — Ranking de Retorno ===")
display(df_gold_ticker[["ticker", "preco_medio", "ultimo_fechamento", "retorno_total_pct", "volatilidade_media", "ranking_retorno"]].sort_values("ranking_retorno"))

In [0]:
print("=== Valores Nulos por Coluna (Gold Resumo Ticker) ===")
nulls_ticker = df_gold_ticker.isnull().sum()
if nulls_ticker.sum() == 0:
    print("Nenhum valor nulo encontrado! ✅")
else:
    display(nulls_ticker)

print("\n=== Valores Nulos por Coluna (Gold Resumo Mensal) ===")
nulls_mensal = df_gold_mensal.isnull().sum()
if nulls_mensal.sum() == 0:
    print("Nenhum valor nulo encontrado! ✅")
else:
    display(nulls_mensal)

In [0]:
# Evolução do preço de fechamento mensal por ticker (Gold)
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Tickers brasileiros
tickers_br = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBAS3.SA", "WEGE3.SA", "B3SA3.SA"]
for t in tickers_br:
    subset = df_gold_mensal[df_gold_mensal["ticker"] == t].sort_values(["ano", "mes"])
    ano_mes = subset["ano"].astype(str) + "-" + subset["mes"].astype(str).str.zfill(2)
    axes[0].plot(ano_mes, subset["fechamento"], label=t, linewidth=1.2)
axes[0].set_title("Evolução do Preço de Fechamento Mensal — Ações Brasileiras", fontsize=14)
axes[0].set_ylabel("Preço (R$)")
axes[0].legend(loc="upper left")
axes[0].tick_params(axis="x", rotation=45)

# Tickers americanos
tickers_us = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META"]
for t in tickers_us:
    subset = df_gold_mensal[df_gold_mensal["ticker"] == t].sort_values(["ano", "mes"])
    ano_mes = subset["ano"].astype(str) + "-" + subset["mes"].astype(str).str.zfill(2)
    axes[1].plot(ano_mes, subset["fechamento"], label=t, linewidth=1.2)
axes[1].set_title("Evolução do Preço de Fechamento Mensal — Ações Americanas", fontsize=14)
axes[1].set_ylabel("Preço (US$)")
axes[1].legend(loc="upper left")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [0]:
# Volume total de negociação por ticker (Gold)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

vol_br = df_gold_ticker[df_gold_ticker["ticker"].isin(tickers_br)].set_index("ticker")["volume_total"].sort_values(ascending=False)
axes[0].barh(vol_br.index, vol_br.values, color="steelblue")
axes[0].set_title("Volume Total — Ações Brasileiras", fontsize=13)
axes[0].set_xlabel("Volume total")

vol_us = df_gold_ticker[df_gold_ticker["ticker"].isin(tickers_us)].set_index("ticker")["volume_total"].sort_values(ascending=False)
axes[1].barh(vol_us.index, vol_us.values, color="coral")
axes[1].set_title("Volume Total — Ações Americanas", fontsize=13)
axes[1].set_xlabel("Volume total")

plt.tight_layout()
plt.show()

In [0]:
# Distribuição dos retornos mensais (Gold)
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
all_tickers = tickers_br + tickers_us
for i, ticker in enumerate(all_tickers):
    ax = axes[i // 3, i % 3]
    subset = df_gold_mensal[df_gold_mensal["ticker"] == ticker]["retorno_mensal_pct"].dropna()
    ax.hist(subset, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(subset.mean(), color="red", linestyle="--", linewidth=1.5, label=f"Média: {subset.mean():.2f}%")
    ax.set_title(ticker, fontsize=12)
    ax.set_xlabel("Retorno mensal (%)")
    ax.legend(fontsize=9)

plt.suptitle("Distribuição dos Retornos Mensais por Ticker", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

# Estatísticas de retorno mensal
print("\n=== Estatísticas de Retorno Mensal ===")
ret_stats = df_gold_mensal.groupby("ticker")["retorno_mensal_pct"].agg(["mean", "std", "min", "max"]).round(3)
ret_stats["sharpe_approx"] = (ret_stats["mean"] / ret_stats["std"]).round(3)
display(ret_stats)

In [0]:
# Matriz de correlação entre os preços de fechamento mensais dos tickers (Gold)
pivot_close = df_gold_mensal.pivot_table(index=["ano", "mes"], columns="ticker", values="fechamento")
corr_matrix = pivot_close.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn", center=0, 
            vmin=-1, vmax=1, square=True, linewidths=0.5, 
            cbar_kws={"label": "Correlação"})
plt.title("Matriz de Correlação — Preço de Fechamento Mensal", fontsize=15)
plt.tight_layout()
plt.show()

In [0]:
# Volatilidade média mensal por ticker (Gold)
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

for t in tickers_br:
    subset = df_gold_mensal[df_gold_mensal["ticker"] == t].sort_values(["ano", "mes"])
    ano_mes = subset["ano"].astype(str) + "-" + subset["mes"].astype(str).str.zfill(2)
    axes[0].plot(ano_mes, subset["volatilidade_media"], label=t, linewidth=1.0)
axes[0].set_title("Volatilidade Média Mensal — Ações Brasileiras", fontsize=14)
axes[0].set_ylabel("Volatilidade (%)")
axes[0].legend(loc="upper left", fontsize=9)
axes[0].tick_params(axis="x", rotation=45)

for t in tickers_us:
    subset = df_gold_mensal[df_gold_mensal["ticker"] == t].sort_values(["ano", "mes"])
    ano_mes = subset["ano"].astype(str) + "-" + subset["mes"].astype(str).str.zfill(2)
    axes[1].plot(ano_mes, subset["volatilidade_media"], label=t, linewidth=1.0)
axes[1].set_title("Volatilidade Média Mensal — Ações Americanas", fontsize=14)
axes[1].set_ylabel("Volatilidade (%)")
axes[1].legend(loc="upper left", fontsize=9)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [0]:
# Retorno mensal a partir da camada Gold
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

for t in tickers_br:
    subset = df_gold_mensal[df_gold_mensal["ticker"] == t].sort_values(["ano", "mes"])
    ano_mes = subset["ano"].astype(str) + "-" + subset["mes"].astype(str).str.zfill(2)
    axes[0].plot(ano_mes, subset["retorno_mensal_pct"], label=t, linewidth=1.0, marker="o", markersize=2)
axes[0].set_title("Retorno Mensal (%) — Ações Brasileiras", fontsize=14)
axes[0].set_ylabel("Retorno (%)")
axes[0].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[0].legend(loc="upper left", fontsize=9)
axes[0].tick_params(axis="x", rotation=45)

for t in tickers_us:
    subset = df_gold_mensal[df_gold_mensal["ticker"] == t].sort_values(["ano", "mes"])
    ano_mes = subset["ano"].astype(str) + "-" + subset["mes"].astype(str).str.zfill(2)
    axes[1].plot(ano_mes, subset["retorno_mensal_pct"], label=t, linewidth=1.0, marker="o", markersize=2)
axes[1].set_title("Retorno Mensal (%) — Ações Americanas", fontsize=14)
axes[1].set_ylabel("Retorno (%)")
axes[1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[1].legend(loc="upper left", fontsize=9)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print("\n=== Retorno Mensal Médio por Ticker ===")
monthly_ret_avg = df_gold_mensal.groupby("ticker")["retorno_mensal_pct"].agg(["mean", "std", "min", "max"]).round(2)
display(monthly_ret_avg)

## Resumo da Análise Exploratória (Camada Gold)

### Pontos de Atenção
- Verificar a consistência do número de registros por ticker (ações brasileiras e americanas podem ter diferentes quantidades de dias de pregão).
- Analisar períodos de alta volatilidade que podem indicar eventos de mercado.
- Correlações entre tickers do mesmo país tendem a ser mais altas.

### Próximos Passos
- Comparar desempenho entre ações brasileiras e americanas (normalização de moeda).
- Análise técnica com médias móveis, RSI, MACD.
- Modelagem preditiva (forecast de preços).
- Backtesting de estratégias de trading.